# [9.3] Emergent Misalignment Detection - Solutions

Reference validation notebook for the section-local benign proxy-drift implementation and committed Pythia hidden-state verification report.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter9_alignment_interpretability"
section = "part3_emergent_misalignment_detection"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_emergent_misalignment_detection.tests as tests
from chapter9_alignment_interpretability.exercises.part3_emergent_misalignment_detection import solutions

In [ ]:
tests.test_proxy_kinds_are_explicit_safe_categories(
    solutions.proxy_kinds_smoke_test,
)
tests.test_drift_detector_report_scores_heldout_logits(
    solutions.drift_detector_report,
)
tests.test_crosscoder_alignment_uses_pearson_correlation(
    solutions.crosscoder_drift_alignment_report,
)
tests.test_mitigation_report_bounds_capability_loss(
    solutions.drift_mitigation_report,
)
tests.test_early_warning_report_compares_detection_steps(
    solutions.early_warning_report,
)
tests.test_detector_smoke_test(solutions.detector_smoke_test)
tests.test_crosscoder_smoke_test(solutions.crosscoder_smoke_test)
tests.test_mitigation_smoke_test(solutions.mitigation_smoke_test)
tests.test_early_warning_smoke_test(solutions.early_warning_smoke_test)
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["proxy_kinds"] == [
    "sycophantic",
    "overconfident",
    "json_only",
    "style_drift",
    "refusal_overgeneralizing",
]
assert contract["detector"]["predicts_heldout_drift"]
assert contract["crosscoder"]["aligns_with_behavior_delta"]
assert contract["mitigation"]["mitigation_passes"]
assert contract["early_warning"]["white_box_catches_earlier"]
contract

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]

assert report["accepted"]
assert gpu["cuda_available"]
assert gpu["preflight_passed"]
assert gpu["model_name"] == "EleutherAI/pythia-70m-deduped"
assert gpu["hf_revision"] == "e93a9faa9c77e5d09219f6c868bfc7a1bd65593c"
assert gpu["hidden_layer"] == -1
assert gpu["generation_used"] is False
assert gpu["detector_accuracy"] == 1.0
assert gpu["predicts_heldout_drift"]
assert gpu["drift_alignment_correlation"] >= 0.7
assert gpu["aligns_with_behavior_delta"]
assert gpu["label_shuffled_detector_accuracy"] <= 0.75
assert gpu["random_direction_accuracy"] <= 0.55
assert gpu["black_box_behavior_proxy_accuracy"] == 1.0
assert gpu["mitigation_drift_delta_reduction"] >= 1.0
assert gpu["mitigation_neutral_delta_shift"] <= 0.1
assert gpu["mitigation_passes"]
assert gpu["train_prompt_count"] == 36
assert gpu["heldout_prompt_count"] == 24
assert gpu["drift_kind_count"] == 5
assert gpu["hidden_state_shape"] == [24, 512]
assert gpu["within_vram_budget"]

{key: gpu[key] for key in [
    "preflight_passed",
    "detector_accuracy",
    "drift_alignment_correlation",
    "label_shuffled_detector_accuracy",
    "random_direction_accuracy",
    "mitigation_drift_delta_reduction",
    "peak_vram_gb",
]}